<a href="https://colab.research.google.com/github/Manrajak/image_face_sort/blob/main/image_face_sort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install deepface

In [ ]:
# .\.venv\Scripts\activate
# python face_sorter_deepface.py

# face_sorter_deepface.py
import os
import shutil
from deepface import DeepFace
import numpy as np
import pickle

# Paths
INPUT_DIR = "Input_images"
OUTPUT_DIR = "sorted_faces"
UNKNOWN_DIR = os.path.join(OUTPUT_DIR, "unrecognized")
DB_PATH = os.path.join(OUTPUT_DIR, "face_db.pkl")

# Create output dirs if not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UNKNOWN_DIR, exist_ok=True)

# Storage for known embeddings {person_id: [embeddings]}
face_db = {}

# Parameters
MODEL_NAME = "ArcFace"            # Strong recognition model
DETECTOR_BACKEND = "retinaface"   # Accurate detector
# Cosine distance threshold for ArcFace. Typical good range: 0.35 - 0.5
# Lower = stricter (fewer false positives), Higher = looser (fewer false negatives)
COSINE_DISTANCE_THRESHOLD = 0.4
MAX_EMB_PER_PERSON = 50  # cap per-person embeddings to control drift/size

def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
    a_norm = a / (np.linalg.norm(a) + 1e-10)
    b_norm = b / (np.linalg.norm(b) + 1e-10)
    cos_sim = np.dot(a_norm, b_norm)
    return 1.0 - cos_sim

def load_db():
    global face_db
    if os.path.exists(DB_PATH):
        try:
            with open(DB_PATH, "rb") as f:
                loaded = pickle.load(f)
                # Ensure numpy arrays
                face_db = {int(k): [np.array(e, dtype=np.float32) for e in v] for k, v in loaded.items()}
        except Exception as e:
            print(f"[WARN] Failed to load DB: {e}. Starting fresh.")
            face_db = {}
    else:
        face_db = {}

def save_db():
    try:
        serializable = {int(k): [np.asarray(e, dtype=np.float32) for e in v] for k, v in face_db.items()}
        with open(DB_PATH, "wb") as f:
            pickle.dump(serializable, f)
    except Exception as e:
        print(f"[WARN] Failed to save DB: {e}")

def get_embedding(img_path):
    try:
        embeddings = DeepFace.represent(
            img_path=img_path,
            model_name=MODEL_NAME,
            detector_backend=DETECTOR_BACKEND,
            enforce_detection=True
        )
        return embeddings
    except Exception as e:
        print(f"[ERROR] {img_path}: {e}")
        return None

def match_person(embedding: np.ndarray):
    """Return best matching person_id if cosine distance <= threshold, else None"""
    best_id = None
    best_dist = float("inf")
    for person_id, embeddings in face_db.items():
        # compute min distance against stored refs
        for ref_vec in embeddings:
            dist = cosine_distance(embedding, ref_vec)
            if dist < best_dist:
                best_dist = dist
                best_id = person_id
    if best_dist <= COSINE_DISTANCE_THRESHOLD:
        return best_id
    return None

def main():
    load_db()
    image_files = [f for f in os.listdir(INPUT_DIR)
                   if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]

    # Initialize person counter from DB or existing folders
    person_count = max(face_db.keys(), default=0)
    for name in os.listdir(OUTPUT_DIR):
        if name.startswith("person_"):
            try:
                pid = int(name.split("_")[-1])
                person_count = max(person_count, pid)
            except ValueError:
                pass

    for img in image_files:
        img_path = os.path.join(INPUT_DIR, img)
        embeddings = get_embedding(img_path)

        if embeddings is None or len(embeddings) == 0:
            dest = os.path.join(UNKNOWN_DIR, img)
            if not os.path.exists(dest):
                shutil.copy2(img_path, dest)
            continue

        matched_persons = set()
        new_person_embeddings = []  # embeddings that didn't match anyone

        for emb in embeddings:
            emb_vec = np.array(emb["embedding"], dtype=np.float32)
            match_id = match_person(emb_vec)
            if match_id is not None:
                matched_persons.add(match_id)
                # update DB with this new sample
                face_db[match_id].append(emb_vec)
                if len(face_db[match_id]) > MAX_EMB_PER_PERSON:
                    # keep most recent MAX_EMB_PER_PERSON
                    face_db[match_id] = face_db[match_id][-MAX_EMB_PER_PERSON:]
            else:
                new_person_embeddings.append(emb_vec)

        # Copy whole image into each matched person's folder
        for pid in matched_persons:
            dest_dir = os.path.join(OUTPUT_DIR, f"person_{pid}")
            os.makedirs(dest_dir, exist_ok=True)
            dest_path = os.path.join(dest_dir, img)
            if not os.path.exists(dest_path):
                shutil.copy2(img_path, dest_path)

        # For each unmatched face, create a new person bucket
        for emb_vec in new_person_embeddings:
            person_count += 1
            new_id = person_count
            dest_dir = os.path.join(OUTPUT_DIR, f"person_{new_id}")
            os.makedirs(dest_dir, exist_ok=True)
            dest_path = os.path.join(dest_dir, img)
            if not os.path.exists(dest_path):
                shutil.copy2(img_path, dest_path)
            face_db[new_id] = [emb_vec]

        # If no matches and no new persons created (shouldn't happen if faces found), nothing to do
        save_db()

if __name__ == "__main__":
    main()
